# Exploratory Analysis: Financial News Sentiment vs. Stock Price Returns

This notebook demonstrates step-by-step data exploration of financial news headlines scored via NLTK VADER and daily stock returns extracted from yfinance.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plot style
sns.set_theme(style="darkgrid")
%matplotlib inline

## 1. Load Master Dataset

In [ ]:
master_path = "../data/processed/master_sentiment_prices.csv"
if not os.path.exists(master_path):
    master_path = "data/processed/master_sentiment_prices.csv"

df = pd.read_csv(master_path)
df['Date'] = pd.to_datetime(df['Date'])
df.head(10)

## 2. Dataset Summary & Descriptive Statistics

In [ ]:
df.describe()

## 3. Correlation Analysis

In [ ]:
from scipy.stats import pearsonr, spearmanr

ret_col = 'Daily_Return_Pct' if 'Daily_Return_Pct' in df.columns else 'Return_Pct'
valid = df.dropna(subset=['Mean_Sentiment', ret_col])

pearson_r, pearson_p = pearsonr(valid['Mean_Sentiment'], valid[ret_col])
spearman_r, spearman_p = spearmanr(valid['Mean_Sentiment'], valid[ret_col])

print(f"Pearson Correlation r  : {pearson_r:.4f} (p-value: {pearson_p:.4f})")
print(f"Spearman Correlation r : {spearman_r:.4f} (p-value: {spearman_p:.4f})")

## 4. Visualization: Sentiment vs Return Scatter Plot

In [ ]:
ret_col = 'Daily_Return_Pct' if 'Daily_Return_Pct' in df.columns else 'Return_Pct'
plt.figure(figsize=(10, 6))
sns.regplot(data=df, x='Mean_Sentiment', y=ret_col, scatter_kws={'alpha': 0.6}, line_kws={'color': 'red'})
plt.title('Daily Compound Sentiment vs Stock Return (%)')
plt.xlabel('Mean Daily Sentiment Score (VADER)')
plt.ylabel('Daily Stock Return (%)')
plt.show()

## 5. Per-Ticker Correlation Breakdown

In [ ]:
ret_col = 'Daily_Return_Pct' if 'Daily_Return_Pct' in df.columns else 'Return_Pct'
for ticker, g in df.groupby('Ticker'):
    if g['Mean_Sentiment'].std() > 0:
        pr, pp = pearsonr(g['Mean_Sentiment'], g[ret_col])
        print(f"{ticker:<12} | Pearson r: {pr:+.4f} (p={pp:.4f}) | N={len(g)}")
    else:
        print(f"{ticker:<12} | Insufficient sentiment variation (all zero/neutral)")